In [1]:
%cd /kaggle/working
!git clone https://github.com/shujaabbasn/ATML.git
%cd /kaggle/working/ATML/pa4
!python -m pip install -q -r requirements.txt
!nvidia-smi

/kaggle/working
Cloning into 'ATML'...
remote: Enumerating objects: 1190, done.
remote: Counting objects: 100% (638/638), done.
remote: Compressing objects: 100% (408/408), done.
remote: Total 1190 (delta 266), reused 594 (delta 226), pack-reused 552 (from 1)
Receiving objects: 100% (1190/1190), 35.76 MiB | 30.69 MiB/s, done.
Resolving deltas: 100% (496/496), done.
/kaggle/working/ATML/pa4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompati

In [2]:
# staff data repo (scripts + bundled files), then merge anything attached from earlier sessions
!git clone -q https://github.com/AbDu11aHHH/atml-pa4-datasets.git data_repo
!python kaggle_runs/attach_inputs.py
# skips files that already exist, so with data_prepare attached this is only a check (C4 streaming is slow otherwise)
!cd data_repo && python scripts/setup_data.py --public && python scripts/check_setup.py

unzip /kaggle/input/notebooks/shujabbasn/data-prepare-run/data_repo.zip
copy /kaggle/input/notebooks/shujabbasn/data-prepare-run/ATML/pa4/data_repo/data -> data_repo/data
merged 2 inputs
revision lock present
tokenizer_config.json: 7.23kB [00:00, 18.7MB/s]
vocab.json: 2.78MB [00:00, 72.5MB/s]
merges.txt: 1.67MB [00:00, 120MB/s]
tokenizer.json: 7.03MB [00:00, 195MB/s]
Public data available locally. Preserve data/local_source_revisions.json for reproducibility.
OK C4 calibration 128
OK C4 development 32
OK C4 validation windows 177
OK WikiText-2 test windows 437
OK HellaSwag examples 2000
OK GSM8K train 500
OK GSM8K test 60
OK prefix banks
OK source snapshot lock
PASS: public assets prepared locally. Model correctness checks remain your experiment responsibility.


In [3]:
!python -m task3_speculative.sampler_tests

toy buggy= False {'statistic': 14.666733721970765, 'p_value': 0.9631284044536529, 'cells': 27, 'min_expected': 599.9999988079072, 'passes': True}
toy buggy= True {'statistic': 17285.33916518043, 'p_value': 0.0, 'cells': 27, 'min_expected': 599.9999988079072, 'rejected': True}
tokenizer_config.json: 7.30kB [00:00, 26.0MB/s]
vocab.json: 2.78MB [00:00, 107MB/s]
merges.txt: 1.67MB [00:00, 138MB/s]
tokenizer.json: 7.03MB [00:00, 192MB/s]
tokenizer_config.json: 7.30kB [00:00, 28.7MB/s]
vocab.json: 2.78MB [00:00, 127MB/s]
merges.txt: 1.67MB [00:00, 139MB/s]
tokenizer.json: 7.03MB [00:00, 191MB/s]
config.json: 100%|█████████████████████████████| 660/660 [00:00<00:00, 4.51MB/s]
I0000 00:00:1795546849.658696     102 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
model.safetensors: 100%|████████████████████| 3.09G/3.09G [00:09<00:00, 329MB/s]
config.json: 100%|█████████████████████████████| 659/659 [00:00<00:00, 4.09MB/s]
model.safetensors: 100%|████████████

In [4]:
!python -m task3_speculative.banks --bank exact_spec

I0000 00:00:1795546916.162285     179 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
exact_spec_original_gamma4 20 / 120
exact_spec_original_gamma4 40 / 120
exact_spec_original_gamma4 60 / 120
exact_spec_original_gamma4 80 / 120
exact_spec_original_gamma4 100 / 120
exact_spec_original_gamma4 120 / 120


In [5]:
# the target sampled alone at temperature 1: the matched reference for the consistency check below
# (the staff teacher bank used Qwen's default repetition_penalty 1.1, so it is not the same sampling law)
!python -m task3_speculative.banks --bank target_plain

I0000 00:00:1795547483.511331     203 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
target_plain 20 / 120
target_plain 40 / 120
target_plain 60 / 120
target_plain 80 / 120
target_plain 100 / 120
target_plain 120 / 120


In [6]:
!python -m task3_speculative.banks --bank consistency

I0000 00:00:1795547994.241159     227 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
spec {'length_ks_statistic': 0.06666666666666667, 'length_ks_p': 0.9538360034954086, 'mean_reference_length': 97.15833333333333, 'mean_other_length': 100.43333333333334, 'nll_wilcoxon_p': 0.27998170550746293, 'mean_reference_nll': 0.343354129413198, 'mean_other_nll': 0.383656948582014, 'nll_pairs': 120, 'repeat_wilcoxon_p': 0.4754818027902459, 'mean_reference_repeat': 0.5967238382831354, 'mean_other_repeat': 0.6027639277054355}
teacher {'length_ks_statistic': 0.11666666666666667, 'length_ks_p': 0.3889566002413223, 'mean_reference_length': 97.15833333333333, 'mean_other_length': 104.66666666666667, 'nll_wilcoxon_p': 7.238781671760464e-11, 'mean_reference_nll': 0.343354129413198, 'mean_other_nll': 0.8818361716566263, 'nll_pairs': 120, 'repeat_wilcoxon_p': 6.6818278517791774e-12, 'mean_reference_repeat': 0.5967238382831354, 'mean_other_repeat': 0.4893937918401915}
or

In [7]:
!python -m task3_speculative.spec_eval --mode speed --drafts original
!python -m task3_speculative.spec_eval --mode quality --drafts original

I0000 00:00:1795548041.274422     251 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
spec_original_g2 0 / 120
spec_original_g2 20 / 120
spec_original_g2 40 / 120
spec_original_g2 60 / 120
spec_original_g2 80 / 120
spec_original_g2 100 / 120
spec_original_g4 0 / 120
spec_original_g4 20 / 120
spec_original_g4 40 / 120
spec_original_g4 60 / 120
spec_original_g4 80 / 120
spec_original_g4 100 / 120
I0000 00:00:1795549743.792411     275 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
quality done target_only
quality done student_only_original


In [8]:
!python -m task3_speculative.distill --step teacher_bank

I0000 00:00:1795552424.984136     299 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
teacher bank 100 / 800
teacher bank 200 / 800
teacher bank 300 / 800
teacher bank 400 / 800
teacher bank 500 / 800
teacher bank 600 / 800
teacher bank 700 / 800
teacher bank 800 / 800


In [9]:
!zip -r /kaggle/working/task3_baseline_results.zip results/task3

  adding: results/task3/ (stored 0%)
  adding: results/task3/banks/ (stored 0%)
  adding: results/task3/banks/exact_spec_original_gamma4.jsonl.gz (deflated 0%)
  adding: results/task3/banks/target_plain.jsonl.gz (deflated 0%)
  adding: results/task3/banks/consistency.json (deflated 85%)
  adding: results/task3/sampler_tests.json (deflated 76%)
  adding: results/task3/speed/ (stored 0%)
  adding: results/task3/speed/spec_original_g2.jsonl (deflated 81%)
  adding: results/task3/speed/spec_original_g2.warmup.json (deflated 21%)
  adding: results/task3/speed/spec_original_g4.warmup.json (deflated 21%)
  adding: results/task3/speed/spec_original_g4.jsonl (deflated 81%)
  adding: results/task3/config_used.json (deflated 63%)
  adding: results/task3/distill/ (stored 0%)
  adding: results/task3/distill/teacher_bank_cost.json (deflated 30%)
  adding: results/task3/distill/teacher_train_responses.jsonl.gz (deflated 0%)
  adding: results/task3/quality/ (stored 0%)
  adding: results/task3/quality/